In [3]:

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FEATURES = ["unit_weight_kNm3", "cohesion_kPa", "friction_angle_deg",
            "slope_angle_deg", "slope_height_m", "pore_pressure_ratio"]
TARGET = "Status"

# --- 1. Load the 80/20 split -------------------------------------------------
train = pd.read_csv("train80_split.csv")
test = pd.read_csv("test_split.csv")

def xy(df):
    X = df[FEATURES].values
    y = (df[TARGET] == "Unstable").astype(int).values  # 1 = Unstable, 0 = Stable
    return X, y

X_train, y_train = xy(train)
X_test, y_test = xy(test)

print(f"Train (80%): {len(X_train)}, Test (20%): {len(X_test)}")

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=FEATURES)

#  2. xgb.cv() over a small hyperparameter grid, picking rounds via ------
#        early stopping WITHIN each fold.
# XGBoost does NOT require feature scaling (it's tree-based) - no scaler
# needed here, same as before.
param_grid = [
    {"max_depth": d, "learning_rate": lr, "subsample": 0.8, "colsample_bytree": 0.8,
     "objective": "binary:logistic", "eval_metric": "logloss", "seed": 42}
    for d in [2, 3, 4]
    for lr in [0.01, 0.05, 0.1]
]

best_logloss = np.inf
best_params = None
best_rounds = None

for params in param_grid:
    cv_results = xgb.cv(
        params, dtrain,
        num_boost_round=500,
        nfold=5,
        stratified=True,
        early_stopping_rounds=20,
        metrics="logloss",
        as_pandas=True,
        seed=42,
        verbose_eval=False,
    )
    mean_logloss = cv_results["test-logloss-mean"].min()
    n_rounds = int(cv_results["test-logloss-mean"].idxmin()) + 1
    if mean_logloss < best_logloss:
        best_logloss = mean_logloss
        best_params = params
        best_rounds = n_rounds

print(f"\nBest hyperparameters (via 5-fold CV on the 80% training pool): "
      f"max_depth={best_params['max_depth']}, learning_rate={best_params['learning_rate']}")
print(f"Best number of boosting rounds (from CV, with early stopping per fold): {best_rounds}")
print(f"Best CV log-loss: {best_logloss:.4f}")

# --- 3. Fit the FINAL model on the FULL 80% training pool -------------------
#        (no data held back - this is the point of moving to 80/20 + CV)
model = xgb.XGBClassifier(
    n_estimators=best_rounds,
    max_depth=best_params["max_depth"],
    learning_rate=best_params["learning_rate"],
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="logloss", random_state=42,
)
model.fit(X_train, y_train)

# --- 4. Evaluate ONCE on the held-out TEST set -------------------------------
pred = model.predict(X_test)
pred_proba = model.predict_proba(X_test)[:, 1]

print(f"\nFinal held-out TEST accuracy: {accuracy_score(y_test, pred):.3f}")
print(f"Final held-out TEST ROC-AUC: {roc_auc_score(y_test, pred_proba):.3f}")
print("\nClassification report (test set):")
print(classification_report(y_test, pred, target_names=["Stable", "Unstable"]))
print("Confusion matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test, pred))

# --- 5. Feature importance ---------------------------------------------------
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(7, 4.5))
importances.plot.barh(ax=ax, color="#2a6f97")
ax.set_xlabel("XGBoost feature importance (gain-based)")
ax.set_title("XGBoost - key instability indicators\n(80% training pool, 5-fold CV-selected hyperparameters)")
fig.tight_layout()
fig.savefig("xgboost_feature_importance.png", dpi=150)
print("\nSaved xgboost_feature_importance.png")

# --- 6. Save the trained model for reuse -------------------------------------
model.save_model("xgboost_slope_model.json")
print("Saved trained model -> xgboost_slope_model.json")

Train (80%): 443, Test (20%): 111

Best hyperparameters (via 5-fold CV on the 80% training pool): max_depth=4, learning_rate=0.1
Best number of boosting rounds (from CV, with early stopping per fold): 82
Best CV log-loss: 0.3608

Final held-out TEST accuracy: 0.838
Final held-out TEST ROC-AUC: 0.912

Classification report (test set):
              precision    recall  f1-score   support

      Stable       0.82      0.85      0.84        54
    Unstable       0.85      0.82      0.84        57

    accuracy                           0.84       111
   macro avg       0.84      0.84      0.84       111
weighted avg       0.84      0.84      0.84       111

Confusion matrix (rows=actual, cols=predicted):
[[46  8]
 [10 47]]

Saved xgboost_feature_importance.png
Saved trained model -> xgboost_slope_model.json


In [ ]:

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FEATURES = ["unit_weight_kNm3", "cohesion_kPa", "friction_angle_deg",
            "slope_angle_deg", "slope_height_m", "pore_pressure_ratio"]
TARGET = "Status"

# --- 1. Load the 80/20 split -------------------------------------------------
train = pd.read_csv("train80_split.csv")
test = pd.read_csv("test_split.csv")

def xy(df):
    X = df[FEATURES].values
    y = (df[TARGET] == "Unstable").astype(int).values  # 1 = Unstable, 0 = Stable
    return X, y

X_train, y_train = xy(train)
X_test, y_test = xy(test)

print(f"Train (80%): {len(X_train)}, Test (20%): {len(X_test)}")

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=FEATURES)

# --- 2. xgb.cv() over a small hyperparameter grid, picking rounds via ------
#        early stopping WITHIN each fold.
# XGBoost does NOT require feature scaling (it's tree-based) - no scaler
# needed here, same as before.
param_grid = [
    {"max_depth": d, "learning_rate": lr, "subsample": 0.8, "colsample_bytree": 0.8,
     "objective": "binary:logistic", "eval_metric": "logloss", "seed": 42}
    for d in [2, 3, 4]
    for lr in [0.01, 0.05, 0.1]
]

best_logloss = np.inf
best_params = None
best_rounds = None

for params in param_grid:
    cv_results = xgb.cv(
        params, dtrain,
        num_boost_round=500,
        nfold=5,
        stratified=True,
        early_stopping_rounds=20,
        metrics="logloss",
        as_pandas=True,
        seed=42,
        verbose_eval=False,
    )
    mean_logloss = cv_results["test-logloss-mean"].min()
    n_rounds = int(cv_results["test-logloss-mean"].idxmin()) + 1
    if mean_logloss < best_logloss:
        best_logloss = mean_logloss
        best_params = params
        best_rounds = n_rounds

print(f"\nBest hyperparameters (via 5-fold CV on the 80% training pool): "
      f"max_depth={best_params['max_depth']}, learning_rate={best_params['learning_rate']}")
print(f"Best number of boosting rounds (from CV, with early stopping per fold): {best_rounds}")
print(f"Best CV log-loss: {best_logloss:.4f}")

# --- 3. Fit the FINAL model on the FULL 80% training pool -------------------
#        (no data held back - this is the point of moving to 80/20 + CV)
model = xgb.XGBClassifier(
    n_estimators=best_rounds,
    max_depth=best_params["max_depth"],
    learning_rate=best_params["learning_rate"],
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="logloss", random_state=42,
)
model.fit(X_train, y_train)

# --- 4. Evaluate ONCE on the held-out TEST set -------------------------------
pred = model.predict(X_test)
pred_proba = model.predict_proba(X_test)[:, 1]

print(f"\nFinal held-out TEST accuracy: {accuracy_score(y_test, pred):.3f}")
print(f"Final held-out TEST ROC-AUC: {roc_auc_score(y_test, pred_proba):.3f}")
print("\nClassification report (test set):")
print(classification_report(y_test, pred, target_names=["Stable", "Unstable"]))
print("Confusion matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test, pred))

# --- 5. Feature importance ---------------------------------------------------
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(7, 4.5))
importances.plot.barh(ax=ax, color="#2a6f97")
ax.set_xlabel("XGBoost feature importance (gain-based)")
ax.set_title("XGBoost - key instability indicators\n(80% training pool, 5-fold CV-selected hyperparameters)")
fig.tight_layout()
fig.savefig("xgboost_feature_importance.png", dpi=150)
print("\nSaved xgboost_feature_importance.png")

# --- 6. Save the trained model for reuse -------------------------------------
model.save_model("xgboost_slope_model.json")
print("Saved trained model -> xgboost_slope_model.json")

# --- 7. Additional check: test set + the 2 real, finalised UCT cases -------
#         (test_split_with_uct.csv = test_split.csv + 2 UCT rows, tagged by
#         a 'source' column). Reported separately from step 4 above - this
#         does NOT replace the clean test-set number, it supplements it.
test_uct = pd.read_csv("test_split_with_uct.csv")
X_test_uct, y_test_uct = xy(test_uct)

pred_uct_all = model.predict(X_test_uct)
print(f"\nTest + real UCT cases ({len(test_uct)} rows) accuracy: "
      f"{accuracy_score(y_test_uct, pred_uct_all):.3f}")

uct_only = test_uct[test_uct["source"] == "UCT case data (finalised)"].copy()
X_uct, y_uct = xy(uct_only)
pred_uct = model.predict(X_uct)
pred_uct_label = np.where(pred_uct == 1, "Unstable", "Stable")

print("\nReal UCT case predictions (never seen during training or CV):")
for case_no, actual, predicted in zip(uct_only["Case No"], uct_only["Status"], pred_uct_label):
    match = "correct" if actual == predicted else "INCORRECT"
    print(f"  {case_no}: actual={actual}, predicted={predicted} ({match})")

Train (80%): 443, Test (20%): 111

Best hyperparameters (via 5-fold CV on the 80% training pool): max_depth=4, learning_rate=0.1
Best number of boosting rounds (from CV, with early stopping per fold): 82
Best CV log-loss: 0.3608

Final held-out TEST accuracy: 0.838
Final held-out TEST ROC-AUC: 0.912

Classification report (test set):
              precision    recall  f1-score   support

      Stable       0.82      0.85      0.84        54
    Unstable       0.85      0.82      0.84        57

    accuracy                           0.84       111
   macro avg       0.84      0.84      0.84       111
weighted avg       0.84      0.84      0.84       111

Confusion matrix (rows=actual, cols=predicted):
[[46  8]
 [10 47]]

Saved xgboost_feature_importance.png
Saved trained model -> xgboost_slope_model.json

Test + real UCT cases (113 rows) accuracy: 0.841

Real UCT case predictions (never seen during training or CV):
  UCT South side: actual=Unstable, predicted=Unstable (correct)
  UCT N

In [5]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

labels = ["Stable", "Unstable"]
cm = confusion_matrix(y_test, pred)            # rows = actual, cols = predicted

fig, ax = plt.subplots(figsize=(5, 4.5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")

ax.set_title("XGBoost confusion matrix (test set, n=111)")
ax.set_xlabel("Predicted label")
ax.set_ylabel("Actual label")
fig.tight_layout()
fig.savefig("xgboost_confusion_matrix.png", dpi=300)
plt.show()

/var/folders/7s/ctpq_jjj4lx6zmyqfy7bvsjm0000gn/T/ipykernel_17869/2429992205.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

labels = ["Stable", "Unstable"]
cm = confusion_matrix(y_test_uct, pred_uct_all)    # rows = actual, cols = predicted

fig, ax = plt.subplots(figsize=(5, 4.5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")

ax.set_title("XGBoost confusion matrix (test set + UCT cases, n=113)")
ax.set_xlabel("Predicted label")
ax.set_ylabel("Actual label")
fig.tight_layout()
fig.savefig("xgboost_confusion_matrix_with_uct.png", dpi=300)
plt.show()

/var/folders/7s/ctpq_jjj4lx6zmyqfy7bvsjm0000gn/T/ipykernel_17869/1173816494.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
